# Anomaly Detection - SCIP Semantic Index Exploration

This notebook explores anomaly detection results for SCIP-indexed code at three abstraction levels:
`SemanticCodeIndexInternalType`, `SemanticCodeIndexModule`, and `SemanticCodeIndexArtifact`.
It shows anomaly scores, top Bridge and Outlier nodes, and feature distributions.

<br>

### References
- [jqassistant](https://jqassistant.org)
- [Neo4j Python Driver](https://neo4j.com/docs/api/python-driver/current)
- [SCIP Semantic Code Intelligence Protocol](https://github.com/sourcegraph/scip)

In [ ]:
import os
import typing

from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plot

In [ ]:
#The following cell uses the build-in %html "magic" to override the CSS style for tables to a much smaller size.
#This is especially needed for PDF export of tables with multiple columns.

In [ ]:
%%html
<style>
/* CSS style for smaller dataframe tables. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

In [ ]:
from sys import version as python_version
print('Python version: {}'.format(python_version))

from numpy import __version__ as numpy_version
print('numpy version: {}'.format(numpy_version))

from pandas import __version__ as pandas_version
print('pandas version: {}'.format(pandas_version))

from matplotlib import __version__ as matplotlib_version
print('matplotlib version: {}'.format(matplotlib_version))

from neo4j import __version__ as neo4j_version
print('neo4j version: {}'.format(neo4j_version))

In [ ]:
# Please set the environment variable "NEO4J_INITIAL_PASSWORD" in your shell
# before starting jupyter notebook to provide the password for the user "neo4j".
# It is not recommended to hardcode the password into jupyter notebook for security reasons.
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    uri="bolt://localhost:7687",
    auth=("neo4j", os.environ.get("NEO4J_INITIAL_PASSWORD"))
)
driver.verify_connectivity()

In [ ]:
def query_cypher_to_data_frame(query: typing.LiteralString, parameters: typing.Optional[typing.Dict[str, typing.Any]] = None) -> pd.DataFrame:
    records, summary, keys = driver.execute_query(query, parameters_=parameters)
    return pd.DataFrame([record.values() for record in records], columns=keys)

## 1 - SCIP InternalType Anomaly Detection

Anomaly detection results for `SemanticCodeIndexInternalType` nodes (individual types/classes/interfaces indexed by SCIP).
Anomaly scores, Bridge, and Outlier archetypes reveal structurally unusual types in the dependency graph.

In [ ]:
scip_internal_type_count: pd.DataFrame = query_cypher_to_data_frame("""
    MATCH (n:SemanticCodeIndexInternalType)
    RETURN count(n) AS totalTypes
          ,count(n.anomalyScore)      AS typesWithAnomalyScore
          ,count(n.anomalyBridgeRank) AS bridgeNodes
          ,count(n.anomalyOutlierRank) AS outlierNodes
""")
display(scip_internal_type_count)

### 1.1 Anomaly Score Distribution

Isolation Forest assigns negative anomaly scores to anomalous nodes. More negative = more anomalous.

In [ ]:
scip_internal_type_anomaly_scores: pd.DataFrame = query_cypher_to_data_frame("""
    MATCH (n:SemanticCodeIndexInternalType)
    WHERE n.anomalyScore IS NOT NULL
    RETURN n.fqn         AS fqn
          ,n.anomalyScore AS anomalyScore
          ,n.anomalyLabel AS anomalyLabel
    ORDER BY n.anomalyScore ASC
    LIMIT 200
""")

if not scip_internal_type_anomaly_scores.empty:
    scip_internal_type_anomaly_scores['anomalyScore'].plot(
        kind='hist',
        bins=30,
        title='SCIP InternalType Anomaly Score Distribution',
        xlabel='Anomaly Score',
        ylabel='Count'
    )
    plot.tight_layout()
    plot.show()
else:
    print('No anomaly scores found for SemanticCodeIndexInternalType. Run anomalyDetectionPython.sh first.')

### 1.2 Features Overview

Overview of collected graph features for SCIP InternalType nodes.

In [ ]:
scip_internal_type_features: pd.DataFrame = query_cypher_to_data_frame("""
    MATCH (n:SemanticCodeIndexInternalType)
    WHERE n.centralityPageRank        IS NOT NULL
      AND n.centralityBetweenness     IS NOT NULL
      AND n.abstractness              IS NOT NULL
    RETURN n.fqn                                    AS fqn
          ,n.incomingDependencies                   AS incomingDependencies
          ,n.outgoingDependencies                   AS outgoingDependencies
          ,n.centralityPageRank                     AS pageRank
          ,n.centralityArticleRank                  AS articleRank
          ,n.centralityBetweenness                  AS betweenness
          ,n.communityLocalClusteringCoefficient    AS clusteringCoefficient
          ,n.abstractness                           AS abstractness
    ORDER BY n.centralityPageRank DESC
    LIMIT 50
""")
display(scip_internal_type_features)

### 1.3 Top Bridge Nodes

Bridge anomalies are types with an unusually high normalized Page-to-Article-Rank difference — they receive more dependencies than they depend on, acting as structural bridges.

In [ ]:
scip_internal_type_top_bridge: pd.DataFrame = query_cypher_to_data_frame("""
    MATCH (n:SemanticCodeIndexInternalType)
    WHERE n.anomalyBridgeRank IS NOT NULL
    RETURN n.anomalyBridgeRank                                                      AS bridgeRank
          ,n.fqn                                                                    AS fqn
          ,n.centralityPageRankToArticleRankDifference                              AS pageToArticleRankDiff
          ,n.incomingDependencies                                                   AS incomingDependencies
          ,n.outgoingDependencies                                                   AS outgoingDependencies
    ORDER BY n.anomalyBridgeRank ASC
    LIMIT 20
""")
display(scip_internal_type_top_bridge)

### 1.4 Top Outlier Nodes

Outlier anomalies are types that don't fit well into any cluster — they have a high normalized distance to their cluster medoid and a low HDBSCAN membership probability.

In [ ]:
scip_internal_type_top_outlier: pd.DataFrame = query_cypher_to_data_frame("""
    MATCH (n:SemanticCodeIndexInternalType)
    WHERE n.anomalyOutlierRank IS NOT NULL
    RETURN n.anomalyOutlierRank                                                      AS outlierRank
          ,n.fqn                                                                     AS fqn
          ,n.clusteringHDBSCANNormalizedDistanceToMedoid                             AS distanceToMedoid
          ,1.0 - n.clusteringHDBSCANProbability                                      AS approximateOutlierScore
          ,n.incomingDependencies                                                    AS incomingDependencies
          ,n.outgoingDependencies                                                    AS outgoingDependencies
    ORDER BY n.anomalyOutlierRank ASC
    LIMIT 20
""")
display(scip_internal_type_top_outlier)

## 2 - SCIP Module Anomaly Detection

Anomaly detection results for `SemanticCodeIndexModule` nodes (files or modules indexed by SCIP).

In [ ]:
scip_module_count: pd.DataFrame = query_cypher_to_data_frame("""
    MATCH (m:SemanticCodeIndexModule)
    RETURN count(m) AS totalModules
          ,count(m.anomalyScore)       AS modulesWithAnomalyScore
          ,count(m.anomalyBridgeRank)  AS bridgeModules
          ,count(m.anomalyOutlierRank) AS outlierModules
""")
display(scip_module_count)

In [ ]:
scip_module_top_anomalies: pd.DataFrame = query_cypher_to_data_frame("""
    MATCH (m:SemanticCodeIndexModule)
    WHERE m.anomalyScore IS NOT NULL
    RETURN m.fqn          AS fqn
          ,m.anomalyScore AS anomalyScore
          ,m.anomalyLabel AS anomalyLabel
          ,coalesce(m.anomalyBridgeRank,  -1) AS bridgeRank
          ,coalesce(m.anomalyOutlierRank, -1) AS outlierRank
    ORDER BY m.anomalyScore ASC
    LIMIT 20
""")
display(scip_module_top_anomalies)

## 3 - SCIP Artifact Anomaly Detection

Anomaly detection results for `SemanticCodeIndexArtifact` nodes (top-level artifacts indexed by SCIP).
Requires at least 15 artifact nodes with `referenceCount` on their `DEPENDS_ON` edges.

In [ ]:
scip_artifact_count: pd.DataFrame = query_cypher_to_data_frame("""
    MATCH (a:SemanticCodeIndexArtifact)
    RETURN count(a) AS totalArtifacts
          ,count(a.anomalyScore)       AS artifactsWithAnomalyScore
          ,count(a.anomalyBridgeRank)  AS bridgeArtifacts
          ,count(a.anomalyOutlierRank) AS outlierArtifacts
""")
display(scip_artifact_count)

In [ ]:
scip_artifact_top_anomalies: pd.DataFrame = query_cypher_to_data_frame("""
    MATCH (a:SemanticCodeIndexArtifact)
    WHERE a.anomalyScore IS NOT NULL
    RETURN a.fqn          AS fqn
          ,a.anomalyScore AS anomalyScore
          ,a.anomalyLabel AS anomalyLabel
          ,coalesce(a.anomalyBridgeRank,  -1) AS bridgeRank
          ,coalesce(a.anomalyOutlierRank, -1) AS outlierRank
    ORDER BY a.anomalyScore ASC
    LIMIT 20
""")

if scip_artifact_top_anomalies.empty:
    print('No anomaly data for SemanticCodeIndexArtifact. Below threshold (15 nodes) or pipeline not yet run.')
else:
    display(scip_artifact_top_anomalies)